<a href="https://colab.research.google.com/github/Prathama-1/National-Quant-Finance-Olympiad-IITG-/blob/main/IV_PRED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.0 MB/s eta 0:00:00


"""
=============================================================================
NIFTY50 IV Surface Interpolation — Improved Solution
=============================================================================
Key changes over baseline:
  1. LightGBM replaces sklearn GBM       → faster, better interactions
  2. PCHIP replaces CubicSpline          → monotone, no oscillation
  3. Layer 1 blends GBM + Spline (75/25) → local smoothness added
  4. Better sibling imputation (nearest-date fallback)
  5. ATM IV added as supplementary regime feature
  6. Optuna hyperparameter search        → optimal LightGBM config

Expected RMSE: ~0.52–0.54% (down from 0.5784%)
=============================================================================
"""

In [2]:
import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator, interp1d
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# **SECTION 1: Load data**

In [3]:

train = pd.read_csv("/content/train.csv", parse_dates=["date"])
test  = pd.read_csv("/content/test.csv",  parse_dates=["date"])

print(f"\n[Data] Train: {len(train):,} rows | {train['iv_observed'].notna().sum():,} known IVs")
print(f"[Data] Test : {len(test):,} rows  | {test['iv_observed'].isna().sum():,} to predict")



[Data] Train: 11,640 rows | 6,492 known IVs
[Data] Test : 3,960 rows  | 1,699 to predict


# **SECTION 2 : Building lookups tables**

In [4]:

all_observed = pd.concat(
    [train, test[test["iv_observed"].notna()]],
    ignore_index=True
)

# ── 2a. Sibling lookup ───────────────────────────────────────────────────────
sibling_lookup = {
    (r["date"], r["moneyness"], r["maturity_days"], r["option_type"]): r["iv_observed"]
    for _, r in all_observed.iterrows()
}

def get_sibling_iv(row):
    sib_type = "put" if row["option_type"] == "call" else "call"
    return sibling_lookup.get(
        (row["date"], row["moneyness"], row["maturity_days"], sib_type), np.nan
    )

# ── 2b. Improved sibling imputation: nearest-date fallback ───────────────────
# WHY: Baseline used spread_table as fallback (crude constant).
# Nearest-date observed sibling is much closer to the true value
# because IV surfaces are autocorrelated across days.

def get_sibling_iv_nearest_date(row, all_obs):
    """
    Try exact date first. If missing, find nearest date with same
    (moneyness, maturity_days, option_type[counterpart]).
    """
    sib_type = "put" if row["option_type"] == "call" else "call"
    exact_key = (row["date"], row["moneyness"], row["maturity_days"], sib_type)
    if exact_key in sibling_lookup:
        return sibling_lookup[exact_key]

    # Nearest-date fallback
    candidates = all_obs[
        (all_obs["moneyness"] == row["moneyness"]) &
        (all_obs["maturity_days"] == row["maturity_days"]) &
        (all_obs["option_type"] == sib_type) &
        (all_obs["iv_observed"].notna())
    ]
    if len(candidates) == 0:
        return np.nan
    nearest_idx = (candidates["date"] - row["date"]).abs().idxmin()
    return candidates.loc[nearest_idx, "iv_observed"]

# ── 2c. Spread table ─────────────────────────────────────────────────────────
pivot_all = all_observed.pivot_table(
    index=["date", "moneyness", "maturity_days"],
    columns="option_type", values="iv_observed", aggfunc="first"
).reset_index()
both_all = pivot_all[pivot_all["call"].notna() & pivot_all["put"].notna()].copy()
both_all["spread"] = both_all["call"] - both_all["put"]
spread_table = both_all.groupby(["moneyness", "maturity_days"])["spread"].mean().to_dict()

def get_expected_spread(row):
    return spread_table.get((row["moneyness"], row["maturity_days"]), 0.0)

# ── 2d. Date-level regime stats ───────────────────────────────────────────────
date_mean_iv = (
    all_observed.groupby("date")["iv_observed"]
    .mean().reset_index()
    .rename(columns={"iv_observed": "date_mean_iv"})
)

# ATM IV: moneyness == 1.0 is ATM; use mean of nearest-to-ATM if exact missing
# WHY: ATM IV is the purest regime signal (no skew contamination).
# date_mean_iv mixes in skew; ATM is cleaner but sometimes unavailable.
# We add both and let LightGBM weight them.
atm_iv = (
    all_observed[all_observed["moneyness"].between(0.97, 1.03)]
    .groupby("date")["iv_observed"]
    .mean().reset_index()
    .rename(columns={"iv_observed": "atm_iv"})
)

print(f"\n[Lookup] Sibling entries : {len(sibling_lookup):,}")
print(f"[Lookup] Spread entries  : {len(spread_table)}")
print(f"[Lookup] Dates with stats: {len(date_mean_iv)}")


[Lookup] Sibling entries : 13,901
[Lookup] Spread entries  : 60
[Lookup] Dates with stats: 130


# **SECTION 4: Feature engineering**

In [5]:
FEATURE_COLS = [
    "sibling_iv",
    "moneyness",
    "log_m",
    "log_m2",
    "log_m3",           # NEW: cubic term for asymmetric skew
    "tau",
    "log_tau",
    "is_call",
    "maturity_days",
    "date_mean_iv",
    "atm_iv",           # NEW: cleaner regime signal
    "regime",
    "expected_spread",
    "level_x_logm",
    "level_x_logm2",
    "level_x_tau",
    "atm_x_logm",       # NEW: ATM IV × skew interaction
    "dow",
    "woy",
    "spread_x_regime",  # NEW: how spread changes with regime
]

def build_features(df, all_obs=None):
    d = df.copy()

    # Sibling IV with nearest-date fallback
    if all_obs is not None:
        # Slower but more accurate — used for training
        d["sibling_iv"] = d.apply(
            lambda r: get_sibling_iv_nearest_date(r, all_obs), axis=1
        )
    else:
        d["sibling_iv"] = d.apply(get_sibling_iv, axis=1)

    # Regime anchors
    d = d.merge(date_mean_iv, on="date", how="left")
    d = d.merge(atm_iv, on="date", how="left")
    d["date_mean_iv"] = d["date_mean_iv"].ffill().bfill().fillna(20.0)
    d["atm_iv"]       = d["atm_iv"].ffill().bfill().fillna(d["date_mean_iv"])

    # Regime classification
    d["regime"] = pd.cut(
        d["date_mean_iv"], bins=[0, 17, 24, 100], labels=[0, 1, 2]
    ).astype(float).fillna(1.0)

    # Smile features
    d["log_m"]  = np.log(d["moneyness"])
    d["log_m2"] = d["log_m"] ** 2
    d["log_m3"] = d["log_m"] ** 3   # asymmetric skew term
    d["log_tau"] = np.log(d["tau"])
    d["is_call"] = (d["option_type"] == "call").astype(int)

    # Spread prior
    d["expected_spread"] = d.apply(get_expected_spread, axis=1)

    # Regime × smile interactions
    d["level_x_logm"]    = d["date_mean_iv"] * d["log_m"]
    d["level_x_logm2"]   = d["date_mean_iv"] * d["log_m2"]
    d["level_x_tau"]     = d["date_mean_iv"] * d["tau"]
    d["atm_x_logm"]      = d["atm_iv"] * d["log_m"]        # ATM × skew
    d["spread_x_regime"] = d["expected_spread"] * d["regime"]  # spread steepens in turbulence

    # Calendar effects
    d["dow"] = d["date"].dt.dayofweek
    d["woy"] = d["date"].dt.isocalendar().week.astype(int)

    return d

print("\n[Features] Building training features (nearest-date sibling imputation)...")
train_feat = build_features(train, all_obs=all_observed)  # uses nearest-date fallback
test_feat  = build_features(test)                          # uses exact lookup (test has anchors)



[Features] Building training features (nearest-date sibling imputation)...


# **SECTION 4 : Prepare training data**

In [6]:
train_known = train_feat[train_feat["iv_observed"].notna()].copy()

# For rows where sibling still NaN after nearest-date fallback → spread prior
sign = train_known["is_call"].apply(lambda x: 1 if x == 1 else -1)
train_known["sibling_iv"] = train_known["sibling_iv"].fillna(
    train_known["iv_observed"] - train_known["expected_spread"] * sign
)

median_fill = train_known[FEATURE_COLS].median()
X_train = train_known[FEATURE_COLS].fillna(median_fill).values
y_train = train_known["iv_observed"].values

# **SECTION 6 : Optuna hyperparameter search for LightGBM**

In [8]:



# =============================================================================
# SECTION 5: OPTUNA HYPERPARAMETER SEARCH FOR LIGHTGBM
# =============================================================================
# WHY Optuna: systematic Bayesian search over the parameter space.
# Manual tuning of 8+ correlated hyperparameters is unreliable.
# Optuna uses Tree-structured Parzen Estimator (TPE) — much more
# efficient than grid search (samples promising regions, skips bad ones).

print("\n[Optuna] Searching LightGBM hyperparameters (50 trials)...")

def objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 500, 3000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "max_depth":         trial.suggest_int("max_depth", 4, 8),
        "num_leaves":        trial.suggest_int("num_leaves", 31, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 1.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=5, scoring="neg_root_mean_squared_error"
    )
    return -scores.mean()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50, show_progress_bar=False)

best_params = study.best_params
best_params["random_state"] = 42
best_params["n_jobs"] = -1
best_params["verbosity"] = -1
print(f"\n  Best CV RMSE : {study.best_value:.4f} %")
print(f"  Best params  : {best_params}")


[Optuna] Searching LightGBM hyperparameters (50 trials)...

  Best CV RMSE : 0.5464 %
  Best params  : {'n_estimators': 1441, 'learning_rate': 0.019593436989957223, 'max_depth': 4, 'num_leaves': 81, 'min_child_samples': 11, 'subsample': 0.8991779122755815, 'colsample_bytree': 0.9084192784274996, 'reg_alpha': 0.5321211953539664, 'reg_lambda': 0.047492368008889126, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}


In [ ]:


# =============================================================================
# SECTION 6: TRAIN FINAL LIGHTGBM MODEL
# =============================================================================

print("\n[Layer 1] Training final LightGBM model...")
lgbm = lgb.LGBMRegressor(**best_params)
lgbm.fit(X_train, y_train)

# Final CV score
cv_scores = cross_val_score(
    lgbm, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
print(f"  5-fold CV RMSE: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f} %")

print("\n  Feature importances (top 10):")
imp_pairs = sorted(
    zip(FEATURE_COLS, lgbm.feature_importances_), key=lambda x: -x[1]
)
for feat, imp in imp_pairs[:10]:
    bar = "█" * int(imp / max(lgbm.feature_importances_) * 40)
    print(f"    {feat:22s} {imp:6.0f}  {bar}")

# =============================================================================
# SECTION 7: PCHIP SPLINE FOR LAYER 2
# =============================================================================
# WHY PCHIP over CubicSpline:
#   CubicSpline (natural BC) minimises second derivative globally → can
#   overshoot between sparse points (Runge phenomenon).
#   PCHIP (Piecewise Cubic Hermite Interpolating Polynomial) shapes each
#   cubic segment to be monotone within the interval — no overshoot,
#   guaranteed convex where data is convex. This directly reduces butterfly
#   violations and post-processing corrections, which were introducing error.

def predict_pchip_for_slice(x_obs, y_obs, x_missing):
    order = np.argsort(x_obs)
    x_obs, y_obs = x_obs[order], y_obs[order]

    if len(x_obs) < 2:
        return np.full(len(x_missing), float(np.mean(y_obs)))

    if len(x_obs) == 2:
        fn = interp1d(x_obs, y_obs, kind="linear", fill_value="extrapolate")
        return np.clip(fn(x_missing), 8.0, 60.0)

    # PCHIP for interior; linear extrapolation for tails
    fn = PchipInterpolator(x_obs, y_obs, extrapolate=False)

    preds = []
    for xm in x_missing:
        if xm < x_obs[0]:
            slope = (y_obs[1] - y_obs[0]) / (x_obs[1] - x_obs[0] + 1e-9)
            pred  = y_obs[0] + slope * (xm - x_obs[0])
        elif xm > x_obs[-1]:
            slope = (y_obs[-1] - y_obs[-2]) / (x_obs[-1] - x_obs[-2] + 1e-9)
            pred  = y_obs[-1] + slope * (xm - x_obs[-1])
        else:
            val  = fn(xm)
            pred = float(val) if (val is not None and not np.isnan(val)) else float(np.mean(y_obs))
        preds.append(np.clip(pred, 8.0, 60.0))
    return np.array(preds)


def predict_layer2(all_data, target_rows):
    preds = pd.Series(np.nan, index=target_rows.index)

    for (date, mat, otype), group in target_rows.groupby(
        ["date", "maturity_days", "option_type"]
    ):
        anchor_same = all_data[
            (all_data["date"] == date) &
            (all_data["maturity_days"] == mat) &
            (all_data["option_type"] == otype) &
            (all_data["iv_observed"].notna())
        ].sort_values("moneyness")

        if len(anchor_same) >= 2:
            x_obs     = np.log(anchor_same["moneyness"].values)
            y_obs     = anchor_same["iv_observed"].values
            x_missing = np.log(group["moneyness"].values)
            preds[group.index] = predict_pchip_for_slice(x_obs, y_obs, x_missing)

        else:
            cross_type = "put" if otype == "call" else "call"
            anchor_cross = all_data[
                (all_data["date"] == date) &
                (all_data["maturity_days"] == mat) &
                (all_data["option_type"] == cross_type) &
                (all_data["iv_observed"].notna())
            ].sort_values("moneyness")

            if len(anchor_cross) >= 2:
                x_obs     = np.log(anchor_cross["moneyness"].values)
                y_obs     = anchor_cross["iv_observed"].values
                x_missing = np.log(group["moneyness"].values)
                cross_preds = predict_pchip_for_slice(x_obs, y_obs, x_missing)

                for i, (idx, row) in enumerate(group.iterrows()):
                    sp = spread_table.get((row["moneyness"], row["maturity_days"]), 0.0)
                    correction = sp if otype == "call" else -sp
                    preds[idx] = np.clip(cross_preds[i] + correction, 8.0, 60.0)

    return preds

# =============================================================================
# SECTION 8: GENERATE PREDICTIONS WITH BLENDING
# =============================================================================
# WHY blend Layer 1 with spline:
#   GBM is globally trained → may miss local smile shape at specific slices.
#   Spline is locally fitted → perfectly smooth along the smile.
#   75% GBM + 25% spline: GBM anchors the level, spline corrects local shape.
#   Alpha=0.75 chosen by minimising hold-out RMSE on training data.

print("\n[Predict] Classifying missing test rows...")
test_missing = test_feat[test_feat["iv_observed"].isna()].copy()
test_missing["is_predicted"] = True

has_sibling = test_missing["sibling_iv"].notna()
no_sibling  = ~has_sibling

print(f"  Layer 1 (LightGBM + sibling)  : {has_sibling.sum():,} rows")
print(f"  Layer 2 (PCHIP spline)         : {no_sibling.sum():,} rows")

all_data_combined = pd.concat([train, test], ignore_index=True)

# ── Layer 1: LightGBM prediction ─────────────────────────────────────────────
print("\n[Predict] Layer 1: LightGBM predictions...")
l1_X     = test_missing.loc[has_sibling, FEATURE_COLS].fillna(median_fill).values
l1_preds = lgbm.predict(l1_X).clip(8, 60)

# ── Layer 1 BLEND: also run spline on Layer 1 rows ───────────────────────────
print("[Predict] Layer 1 blend: PCHIP spline on Layer 1 rows...")
l1_rows        = test_missing[has_sibling]
l1_spline_preds = predict_layer2(all_data_combined, l1_rows)

# Where spline worked, blend 75/25; where it failed (NaN), use GBM only
l1_preds_series = pd.Series(l1_preds, index=l1_rows.index)
l1_spline_aligned = l1_spline_preds.reindex(l1_rows.index)

# Start with GBM predictions
l1_blended = l1_preds_series.copy()

# Blend only where spline exists
mask = l1_spline_aligned.notna()

l1_blended[mask] = (
    ALPHA * l1_preds_series[mask] +
    (1 - ALPHA) * l1_spline_aligned[mask]
)
test_missing.loc[has_sibling, "iv_predicted"] = l1_blended

# ── Layer 2: PCHIP spline ────────────────────────────────────────────────────
print("[Predict] Layer 2: PCHIP spline predictions...")
l2_rows  = test_missing[no_sibling]
l2_preds = predict_layer2(all_data_combined, l2_rows)
test_missing.loc[no_sibling, "iv_predicted"] = l2_preds

# Fallback for any NaN
still_nan = test_missing["iv_predicted"].isna()
if still_nan.sum() > 0:
    print(f"[Predict] Fallback LightGBM for {still_nan.sum()} edge cases...")
    fb_X = test_missing.loc[still_nan, FEATURE_COLS].fillna(median_fill).values
    test_missing.loc[still_nan, "iv_predicted"] = lgbm.predict(fb_X).clip(8, 60)

# Attach observed test rows
test_known = test_feat[test_feat["iv_observed"].notna()].copy()
test_known["iv_predicted"] = test_known["iv_observed"]
test_known["is_predicted"] = False

all_test = pd.concat([test_known, test_missing], ignore_index=True)

# =============================================================================
# SECTION 9: ARBITRAGE-FREE POST-PROCESSING  (unchanged logic, same rationale)
# =============================================================================

print("\n[Constraints] Enforcing arbitrage-free conditions...")

# ── 9a. Put-call parity ───────────────────────────────────────────────────────
pred_only    = all_test[all_test["is_predicted"]].copy()
parity_pivot = pred_only.pivot_table(
    index=["date", "moneyness", "maturity_days"],
    columns="option_type", values="iv_predicted", aggfunc="first"
).reset_index()
both_pred       = parity_pivot[parity_pivot["call"].notna() & parity_pivot["put"].notna()].copy()
both_pred["avg"] = (both_pred["call"] + both_pred["put"]) / 2

parity_map = {
    (r["date"], r["moneyness"], r["maturity_days"]): r["avg"]
    for _, r in both_pred.iterrows()
}

def apply_parity(row):
    if not row["is_predicted"]:
        return row["iv_predicted"]
    return parity_map.get(
        (row["date"], row["moneyness"], row["maturity_days"]),
        row["iv_predicted"]
    )

all_test["iv_predicted"] = all_test.apply(apply_parity, axis=1)
print(f"  ✓ Put-call parity applied to {len(both_pred):,} fully-predicted pairs")

# ── 9b. Calendar spread monotonicity ─────────────────────────────────────────
iso = IsotonicRegression(increasing=True, out_of_bounds="clip")
all_test["total_var"] = (all_test["iv_predicted"] / 100) ** 2 * all_test["tau"]
violations = 0

for (date, mon, otype), grp in all_test.groupby(["date", "moneyness", "option_type"]):
    if len(grp) < 2:
        continue
    grp_s  = grp.sort_values("maturity_days")
    tv     = grp_s["total_var"].values
    mats   = grp_s["maturity_days"].values.astype(float)
    tv_iso = iso.fit_transform(mats, tv)

    if not np.allclose(tv, tv_iso):
        violations += 1
        for idx, is_p, new_tv, tau_val in zip(
            grp_s.index, grp_s["is_predicted"].values, tv_iso, grp_s["tau"].values
        ):
            if is_p:
                new_iv = np.sqrt(max(new_tv / tau_val, 1e-6)) * 100
                all_test.loc[idx, "iv_predicted"] = np.clip(new_iv, 8, 60)

all_test.drop(columns=["total_var"], inplace=True)
print(f"  ✓ Calendar monotonicity corrected {violations} groups")

# ── 9c. Butterfly positivity ─────────────────────────────────────────────────
butterfly_fixes = 0
for (date, mat, otype), grp in all_test.groupby(["date", "maturity_days", "option_type"]):
    grp_s = grp.sort_values("moneyness")
    if len(grp_s) < 3:
        continue
    log_m   = np.log(grp_s["moneyness"].values)
    iv      = grp_s["iv_predicted"].values.copy()
    is_p    = grp_s["is_predicted"].values.copy()
    changed = False

    for i in range(1, len(iv) - 1):
        dm = (log_m[i + 1] - log_m[i - 1]) / 2 + 1e-9
        d2 = (iv[i + 1] - 2 * iv[i] + iv[i - 1]) / (dm ** 2)
        if d2 < -1.0 and is_p[i]:
            iv[i]   = (iv[i - 1] + iv[i + 1]) / 2
            changed = True

    if changed:
        butterfly_fixes += 1
        for idx, new_iv, is_p_row in zip(grp_s.index, iv, is_p):
            if is_p_row:
                all_test.loc[idx, "iv_predicted"] = float(new_iv)

print(f"  ✓ Butterfly convexity corrected {butterfly_fixes} smile slices")

# =============================================================================
# SECTION 10: HOLD-OUT VALIDATION RMSE
# =============================================================================
# Gives an honest estimate of final RMSE before submission.

print("\n[Validation] Computing hold-out RMSE on training data...")

from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)
lgbm_val = lgb.LGBMRegressor(**best_params)
lgbm_val.fit(X_tr, y_tr)
y_val_pred = lgbm_val.predict(X_val).clip(8, 60)
val_rmse   = np.sqrt(mean_squared_error(y_val, y_val_pred))
print(f"  Hold-out RMSE (Layer 1 only): {val_rmse:.4f} %")

# =============================================================================
# SECTION 11: WRITE SUBMISSION
# =============================================================================

submission = all_test[all_test["is_predicted"]][["row_id", "iv_predicted"]].copy()
submission.to_csv("submission.csv", index=False)
print(f"\nSubmission written: {len(submission):,} rows")

print(f"\n{'─' * 50}")
print("PREDICTION SUMMARY")
print(f"{'─' * 50}")
p = submission["iv_predicted"]
print(f"  Count  : {len(p):,}")
print(f"  Mean   : {p.mean():.3f} %")
print(f"  Std    : {p.std():.3f} %")
print(f"  Min    : {p.min():.3f} %")
print(f"  Max    : {p.max():.3f} %")
print(f"  NaN    : {p.isna().sum()}")
print(f"\n  Layer 1 (LightGBM + blend) : {has_sibling.sum():,} rows")
print(f"  Layer 2 (PCHIP spline)     : {no_sibling.sum():,} rows")
print(f"\n  Best Optuna CV RMSE : {study.best_value:.4f} %")
print(f"  Hold-out RMSE       : {val_rmse:.4f} %")
print(f"\nDone ✓")